# RouteHunter

**The app is static.** A CSV file is the single source of truth for the dataset — there is no Contribute/admin pipeline and no in-app way to modify the data. Each CSV you point this at is effectively a version of the app's dataset. To add or correct data, edit the CSV directly and re-run this notebook.

The only thing that happens at runtime beyond reading the CSV is an optional, session-only cache of CASP-predicted routes (Section 4) — it lives in memory for this run and is never written back anywhere.

In [1]:
from routehunter import RouteHunterApp

# This single call is the entire "startup" of the app: load the CSV,
# build the in-memory dataset, done.
app = RouteHunterApp.from_data_dir("rh-data")

print(app.load_report.summary())

[12:42:10] Invalid InChI prefix in generating InChI Key
[12:42:10] Invalid InChI prefix in generating InChI Key


Rows processed : 1392
Targets        : 1390
Unique targets : 1278
Unique papers  : 1171
Errors         : 2

First few errors:
  row 348: Could not derive InChIKey for: 'CC1C=C(C)C(NC(CN2CCN(C3CC(CC4C=CC=CC=4)N(C(C4C=C(C(F)(F)F)C=C(C(F)(F)F)C=4)=O)CC3)CC2)=O)=C([*])C=1'
  row 517: Could not derive InChIKey for: 'CC(OC(NC([*])CC(C1C=CC=CC=1)=O)=O)(C)C'


## Review

In [2]:
print(app.introduction())

RouteHunter -- synthesis route reference lookup (static dataset)

  Search      : give a SMILES, get papers reporting a synthesis route to it.
  Hunter      : browse recently published papers, ranked by predicted
                probability of containing a multi-step synthesis route
                (candidates for you to review and add to the CSV by hand).
  CASP        : predict a route computationally for a target with no
                known literature synthesis (cached for this session only).
  Download    : export the dataset for AI/ML training.

This dataset is loaded from a CSV file; there is no in-app way to
modify it. To add or correct data, edit the CSV and reload.


Current dataset:
  Targets                : 1278
  Papers                 : 1171
  Targets w/ >1 route    : 92
  Cached CASP routes     : 0 (session only)
  Papers by journal:
    Organic Process Research & Development   1171
  Papers by contributor:
    Dmitry Zankov                            1171


## Search

Given a SMILES, return literature papers *and* any CASP-predicted routes cached earlier this session.

Try some molecules with positive search:  
``C#CCOC1=C(C=C(C(=C1)N2C(=O)N3CCCCC3=N2)Cl)Cl``  
``C(O)(C(O)=O)C(C1C=CC=CC=1)NC(C1C=CC=CC=1)=O``

Try some absent molecules:  
``CC(C)Cc1ccc(cc1)C(C)C(=O)O``

In [3]:
result = app.search("C(O)(C(O)=O)C(C1C=CC=CC=1)NC(C1C=CC=CC=1)=O")
print(result.report())

Found 2 paper(s) reporting a route for this molecule:
 - [paper] Some Items of Interest to Process R&amp;D Chemists and Engineers as Selected by Trevor Laird and Stephen A. Hermitage (Organic Process Research & Development, 1999) doi:10.1021/op990089x
 - [paper] Utilization of a Benzoyl Migration To Effect an Expeditious Synthesis of the Paclitaxel C-13 Side Chain (Organic Process Research & Development, 1997) doi:10.1021/op970113b

Found 2 tool(s) predicted routes for this molecule:
 - [AiZynthFinder] This molecule was solved by AiZynthFinder. See predicted routes: please use this tool.
 - [SynPlanner] This molecule was solved by SynPlanner. See predicted routes: please use this tool.


## Monitor

Fetch recent papers, score with a classifier, display ranked. This is **display-only** - nothing here is written into the dataset. If a candidate turns out to be a real route, the way to record it is to add a row to the CSV and reload.

In [4]:
result  = app.monitor()
print(result.message)

5471 paper(s), sorted by predicted route probability.


In [5]:
result.to_dataframe()

,route_prob,journal,title,publication_date,doi
0,98%,Tetrahedron,Stereocontrolled route to a key intermediate f...,"January 1, 1975",10.1016/s0040-4039(00)75203-8
1,98%,Journal of Organic Chemistry,Development of Two Synthetic Routes to a Benzo...,"November 21, 2025",10.1021/acs.joc.5c02114
2,98%,Organic Process Research & Development,An Improved Synthetic Route for Preparative Pr...,"November 2, 2009",10.1021/op900235p
3,98%,Organic Process Research & Development,Development and a Practical Synthesis of the J...,"October 22, 2011",10.1021/op200229j
4,98%,Organic Process Research & Development,Practical Synthesis of a HIV Integrase Inhibitor,"October 29, 2008",10.1021/op800153y
...,...,...,...,...,...
5466,83%,Journal of Organic Chemistry,An Enantioselective Synthesis of the Key Inter...,"March 17, 2014",10.1021/jo500369y
5467,83%,Organic Letters,A Formal Synthesis of (-)-Cephalotaxine,"June 13, 2008",10.1021/ol8010166
5468,83%,Organic Letters,Total Synthesis of (<i>S</i>)-Equol,"November 1, 2006",10.1021/ol0620444
5469,83%,Green Chemistry,Optimization of the use of a chiral bio-based ...,"July 4, 2014",10.1039/c4gc00797b


## Predict

Predict a route computationally. Results are cached in memory for this session (`cache=True` by default) so a later Search this session surfaces them too — but the cache disappears when the notebook restarts; it is never written to the CSV. Uses a stub `CASPEngine` here — swap in a real open-source CASP tool (e.g. AiZynthFinder) via the same `predict_route` interface.

In [6]:
result = app.predict("C(O)(C(O)=O)C(C1C=CC=CC=1)NC(C1C=CC=CC=1)=O")
print(result.to_dataframe().to_string())

            tool probability                                                             url
0  AiZynthFinder         84%                    https://github.com/MolecularAI/aizynthfinder
1     SynPlanner         82%  https://github.com/Laboratoire-de-Chemoinformatique/SynPlanner


## Download

Export the dataset (or a filtered slice) as a flat table for ML training. Every row comes from the CSV — Hunter output never appears here since it's never written into the dataset.

In [7]:
app.download()

,inchikey,canonical_smiles,doi,title,abstract,journal,year,source
0,KJHKTHWMRKYKJE-SUGCFTRWSA-N,Cc1cccc(C)c1OCC(=O)N[C@@H](Cc1ccccc1)[C@@H](O)...,10.1021/op990202j,Synthesis of HIV Protease Inhibitor ABT-378 (L...,A large scale process for the synthesis of HIV...,Organic Process Research & Development,2000,seed
1,XOEMATDHVZOBSG-UHFFFAOYSA-N,C#CCOc1cc(-n2nc3n(c2=O)CCCC3)c(Cl)cc1Cl,10.1021/op9901994,Discovery and Development of a Commercial Synt...,A commercial synthesis of the DuPont herbicide...,Organic Process Research & Development,2001,seed
2,ILMMRHUILQOQGP-UHFFFAOYSA-N,CC(C)(C)c1cc(CC2SCNC2=O)cc(C(C)(C)C)c1O,10.1021/op990197j,The Development of a Manufacturable Synthesis ...,The development of a manufacturable synthesis ...,Organic Process Research & Development,2000,seed
3,MMNZCESXFOCVMF-UHFFFAOYSA-N,O=C1C(C(O)COc2ccc(F)cc2)C(c2ccc(O)cc2)N1c1ccc(...,10.1021/op990196r,A Concise Asymmetric Synthesis of A β-Lactam-B...,"A concise, four-step, asymmetric synthesis of ...",Organic Process Research & Development,2000,seed
4,MRLGCTNJRREZHZ-UHFFFAOYSA-N,O=Cc1cccc(Oc2ccccc2)c1,10.1021/op9901947,Use of Sodium Bromate for Aromatic Bromination...,Sodium bromate is a powerful brominating agent...,Organic Process Research & Development,1999,seed
...,...,...,...,...,...,...,...,...
1385,KXUMNEWSLYWWIA-FPYGCLRLSA-N,C/C=C1\CC2C(=O)Nc3cc(O)c(OC)cc3C(=O)N2C1,10.1021/acs.oprd.0c00009,Total Synthesis of (+)-Oxo-tomaymycin,"(+)-Oxo-tomaymycin, a naturally occurring subs...",Organic Process Research & Development,2020,seed
1386,BLIJXOOIHRSQRB-UHFFFAOYSA-N,CC(Cn1ccc(-c2ccc(C#N)c(Cl)c2)n1)NC(=O)c1cc(C(C...,10.1021/acs.oprd.0c00005,Review of Synthetic Routes and Crystalline For...,This article reviews the synthetic routes and ...,Organic Process Research & Development,2020,seed
1387,WXCXUHSOUPDCQV-UHFFFAOYSA-N,CNC(=O)c1ccc(N2C(=S)N(c3ccc(C#N)c(C(F)(F)F)c3)...,10.1021/acs.oprd.0c00005,Review of Synthetic Routes and Crystalline For...,This article reviews the synthetic routes and ...,Organic Process Research & Development,2020,seed
1388,HJBWBFZLDZWPHF-UHFFFAOYSA-N,CNC(=O)c1ccc(N2C(=S)N(c3cnc(C#N)c(C(F)(F)F)c3)...,10.1021/acs.oprd.0c00005,Review of Synthetic Routes and Crystalline For...,This article reviews the synthetic routes and ...,Organic Process Research & Development,2020,seed
